# [LAB 04] 데이터 다루기 - 4. 결측치 처리

## #01. 준비작업 

### 1. 패키지 설치 

In [39]:
# !pip install --upgrade scikit-learn

### 2. 라이브러리 참조

In [40]:
# import os
# os.environ['PYTHONUTF8'] = '1'
# 이래도 안됨

In [41]:
import numpy as np
from jussam import load_data
from pandas import DataFrame
from sklearn.impute import SimpleImputer

### 3. 데이터 가져오기

In [42]:
origin = load_data("ref_sample")
origin

📚 데이터 정제를 위한 실습용 데이터


,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,NaN,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


## #02. 결측치 확인하기 

### 1. 결측치 여부 확인

In [43]:
empty = origin.isnull()
empty

,kor,eng,math,sic
name,,,,
철수,False,False,False,False
영희,False,False,False,False
민철,True,False,False,False
수현,False,False,False,False
호영,False,False,False,True
영호,False,False,False,False
용식,False,False,True,False
나영,False,False,False,True
석영,False,False,False,False


### 2. 각 열별로 결측치의 수를 확인

In [44]:
empty.sum()

kor     1
eng     0
math    1
sic     2
dtype: int64

## #03. 결측치 소거

### 1. 행 단위 삭제

In [45]:
na1 = origin.dropna()
na1

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
수현,63.000,60,31.000,71.000
영호,80.000,88,91.000,72.000
석영,91.000,90,89.000,80.000


### 2. 열 단위 삭제

In [46]:
na2 = origin.dropna(axis =1)
na2

,eng
name,
철수,77
영희,120
민철,70
수현,60
호영,50
영호,88
용식,88
나영,92
석영,90


### 3. 특정 열에 대해서만 결측치가 존재하는 행 삭제

In [47]:
na3 = origin.dropna(subset=['kor'])
na3 

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


## #04. 결측지 대체 

In [48]:
re_df1 = origin.fillna(value=50)
re_df1

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,50.000,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,50.000
영호,80.000,88,91.000,72.000
용식,82.000,88,50.000,90.000
나영,90.000,92,81.000,50.000
석영,91.000,90,89.000,80.000


### 2. 통계적 값으로 대체(1) - 결측치를 정제할 규칙을 담고 있는 객체 생성

In [49]:
# from sklearn import set_config
# set_config(display='text')  # 기본값 'diagram' → 'text'로 변경

In [50]:
imr = SimpleImputer(missing_values=np.nan, strategy='mean')
imr

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


### 3. 통계적 값으로 대체(2) - 생성된 규칙을 적용

In [51]:
df_imr = imr.fit_transform(origin.values)
df_imr

array([[ 98.        ,  77.        ,  88.        ,  64.        ],
       [ 88.        , 120.        ,  62.        ,  72.        ],
       [ 83.375     ,  70.        ,  83.        ,  79.        ],
       [ 63.        ,  60.        ,  31.        ,  71.        ],
       [ 75.        ,  50.        ,  90.        ,  75.42857143],
       [ 80.        ,  88.        ,  91.        ,  72.        ],
       [ 82.        ,  88.        ,  76.875     ,  90.        ],
       [ 90.        ,  92.        ,  81.        ,  75.42857143],
       [ 91.        ,  90.        ,  89.        ,  80.        ]])

### 4. 통계적 값으로 대체(3) - 결측치 정제 결과를 DataFrame으로 재구성

In [52]:
re_df2 = DataFrame(df_imr, 
                   index = origin.index, # 원본의 인덱스
                   columns = origin.columns # 원본의 컬럼명 
                   )
re_df2

,kor,eng,math,sic
name,,,,
철수,98.000,77.000,88.000,64.000
영희,88.000,120.000,62.000,72.000
민철,83.375,70.000,83.000,79.000
수현,63.000,60.000,31.000,71.000
호영,75.000,50.000,90.000,75.429
영호,80.000,88.000,91.000,72.000
용식,82.000,88.000,76.875,90.000
나영,90.000,92.000,81.000,75.429
석영,91.000,90.000,89.000,80.000


### 5. 특정 컬럼의 값 하나만 처리하기(1) - 컬럼의 값만 추출 후, n행 1열로 재구성

In [53]:
reshape = origin['kor'].values.reshape(-1,1)
reshape

array([[98.],
       [88.],
       [nan],
       [63.],
       [75.],
       [80.],
       [82.],
       [90.],
       [91.]])

### 6. 특정 컬럼의 값 하나만 처리하기 (2) - 특정 컬럼의 값 하나만 처리하기 

In [54]:
reshape_imr = imr.fit_transform(reshape)
reshape_imr

array([[98.   ],
       [88.   ],
       [83.375],
       [63.   ],
       [75.   ],
       [80.   ],
       [82.   ],
       [90.   ],
       [91.   ]])

### 7. 특정 컬럼의 값 하나만 처리하기 (3) - 결측시 처리 결과를 DataFrame의 컬럼에 적용하기

In [55]:
특정컬럼_정제df = origin.copy()
특정컬럼_정제df['kor'] = reshape_imr
특정컬럼_정제df

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,83.375,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


## #1. 연습문제 

In [56]:
from jussam import load_data

### 데이터 불러오기


In [57]:
origin = load_data('month_sales')
origin

📚 데이터 정제(결측치 처리) 연습문제용 데이터

    field     description
--  --------  ----------------
 0  OrderID   주문번호(인덱스)
 1  Product   상품명
 2  Category  상품 카테고리
 3  Price     단가
 4  Quantity  주문 수량



,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1002,Mouse,Electronics,25.000,NaN
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1005,Webcam,Electronics,80.000,NaN
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


### 결측치 확인


In [58]:
df1 = origin.isnull()
df1

,Product,Category,Price,Quantity
OrderID,,,,
1001,False,False,False,False
1002,False,False,False,True
1003,False,False,True,False
1004,False,False,False,False
1005,False,False,False,True
1006,False,False,False,False
1007,False,False,True,False
1008,False,False,False,False
1009,False,False,False,False


In [59]:
df1.sum()

Product     0
Category    0
Price       2
Quantity    3
dtype: int64

In [60]:
df1.isnull().sum()

Product     0
Category    0
Price       0
Quantity    0
dtype: int64

In [61]:
# na1 = origin['Quantity'].dropna()
# na1

### Price 열의 전체 평균가격으로 값을 채워 넣기 

In [62]:
df2 = df1.copy()
df2

,Product,Category,Price,Quantity
OrderID,,,,
1001,False,False,False,False
1002,False,False,False,True
1003,False,False,True,False
1004,False,False,False,False
1005,False,False,False,True
1006,False,False,False,False
1007,False,False,True,False
1008,False,False,False,False
1009,False,False,False,False


In [63]:
df2 = origin.dropna(subset = ['Quantity'])
df2

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [64]:
df3 =df2.copy()
df3

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [65]:
imr2 = SimpleImputer(missing_values=np.nan, strategy='mean')  # 변수명 따로
imr2

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [66]:
df3_reshape = df3['Price'].values.reshape(-1, 1)
df_imr2 = imr2.fit_transform(df3_reshape)
df3['Price'] = df_imr2
df3

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,339.000,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,339.000,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [67]:
origin = df3.copy()
origin

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,339.000,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,339.000,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [68]:
# 파이썬에서 변수는 하나의 값만 담을 수 있어. 새로 할당하면 이전 값은 사라져.
# 연주 코드 순서 보면:
# pythondf3 = df2.copy()          # df3 = DataFrame 📊
# df3 = SimpleImputer(...)  # df3 = Imputer 🔧 (DataFrame 사라짐!)
# origin = df3.copy()       # Imputer에 .copy() → 💥 TypeError
# df3라는 변수에 DataFrame 넣었다가, 바로 다음 줄에서 Imputer로 덮어써버린 거야. 그러니까 마지막 줄에서 df3는 DataFrame이 아니라 Imputer 객체인 거고, Imputer한테 .copy() 부르니까 에러가 난 거지.
# 해결은 변수명만 바꾸면 돼:
# pythondf3 = df2.copy()           # df3 = DataFrame 📊 (유지!)
# imr2 = SimpleImputer(...)  # imr2 = Imputer 🔧 (따로 저장)
# origin = df3.copy()        # df3 아직 DataFrame → ✅ 정상
# 같은 변수명 두 번 쓰지 말고, Imputer는 다른 이름으로 저장하면 끝이야.

In [69]:
## 다시 풀어보기 

In [70]:
# 일단, 데이터를 로드한다. 

In [71]:
from jussam import load_data

In [72]:
# 분석에 필요한, month_salse 데이터를 불러오고, 변수를 지정한다

In [73]:
# 일단 원본 파일

In [74]:
df = load_data('month_sales')
df 

📚 데이터 정제(결측치 처리) 연습문제용 데이터

    field     description
--  --------  ----------------
 0  OrderID   주문번호(인덱스)
 1  Product   상품명
 2  Category  상품 카테고리
 3  Price     단가
 4  Quantity  주문 수량



,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1002,Mouse,Electronics,25.000,NaN
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1005,Webcam,Electronics,80.000,NaN
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [ ]:
# 데이터를 불러왔으니까, 결측지, 누락된 NaN 데이터가 있는지 확인한다. 
# innull로 True, False를 보고, 갯수를 센다. 

In [77]:
df.isnull()

,Product,Category,Price,Quantity
OrderID,,,,
1001,False,False,False,False
1002,False,False,False,True
1003,False,False,True,False
1004,False,False,False,False
1005,False,False,False,True
1006,False,False,False,False
1007,False,False,True,False
1008,False,False,False,False
1009,False,False,False,False


In [78]:
df.sum()

Product                                                   LaptopMouseKeyboardMonitorWebcamDeskChairLampNotebookPen
Category    ElectronicsElectronicsElectronicsElectronicsElectronicsFurnitureFurnitureFurnitureStationeryStationery
Price                                                                                                     1801.000
Quantity                                                                                                   143.000
dtype: object

In [76]:
df.isnull().sum()

Product     0
Category    0
Price       2
Quantity    3
dtype: int64

In [ ]:
# df에서 특정 열의 정보가 누락된 경우, 행을 삭제하기로 한다. 
# Quantity 열의 정보가 누락된 경우, 모든 행을 삭제하기로 한다. 
# 
# subset은 '부분 집합' 또는 '일부'라는 뜻
# dropna는 파이썬 판다스(Pandas)에서 데이터 내 결측치(NaN, Null 등)가 포함된 행(Row)이나 열(Column)을 제거하는 메서드

# df에서, 누락된 특정 집합인, Quantity를 dropna시킨다

In [89]:
df2 = df.dropna(subset=['Quantity'])
df2

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [ ]:
# 가격이 누락 된 경우, Price 열의 전체 평균 가격으로 채워넣는다. 

In [95]:
df2_reshape = df2['Price'].values.reshape(-1,1)

df2_reshape

array([[1200.],
       [  nan],
       [ 300.],
       [ 150.],
       [  nan],
       [  40.],
       [   5.]])

In [97]:
imr = SimpleImputer(missing_values=np.nan, strategy='mean')
df_imr = imr.fit_transform(df2_reshape)
df_imr

array([[1200.],
       [ 339.],
       [ 300.],
       [ 150.],
       [ 339.],
       [  40.],
       [   5.]])

In [98]:
df_imr = imr.fit_transform(df2_reshape)
df_imr


array([[1200.],
       [ 339.],
       [ 300.],
       [ 150.],
       [ 339.],
       [  40.],
       [   5.]])

In [100]:
df3 = df2.copy()
df3

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [102]:
df3['Price'] = df_imr
df3

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,339.000,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,339.000,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000
